In [ ]:


import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
los_asc  = xr.open_dataarray("/scratch/pm4167/China/disp_sbas_CHNA_2016_2019_Asc_B60.nc")
los_desc = xr.open_dataarray("/scratch/pm4167/China/disp_sbas_CHNA_2016_2019_Desc_B60.nc")

# --------------------------------------------------
# Load POI geometry with look vectors
# --------------------------------------------------
poi_geom_asc  = pd.read_csv("/scratch/pm4167/China/Apoi_geometryALLPoints_angles_CHNA_2016_2019_ASC_B60_Apr12.csv")
poi_geom_desc = pd.read_csv("/scratch/pm4167/China/Apoi_geometryALLPoints_angles_CHNA_2016_2019_DESC_B60_Apr12.csv")

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
los_asc  = xr.open_dataarray("/scratch/pm4167/China/disp_sbas_CHNA_2016_2019_Asc_B36.nc")
los_desc = xr.open_dataarray("/scratch/pm4167/China/disp_sbas_CHNA_2014_2019_Desc_B36_Feb26.nc")

# --------------------------------------------------
# Load POI geometry with look vectors
# --------------------------------------------------
poi_geom_asc  = pd.read_csv("/scratch/pm4167/China/Apoi_geometryALLPoints_angles_CHNA_2016_2019_ASC_B36_Feb26.csv")
poi_geom_desc = pd.read_csv("/scratch/pm4167/China/Apoi_geometryALLPoints_angles_CHNA_2016_2019_DESC_B36_Feb26.csv")

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


# --------------------------------------------------
# Vertical + East solver
# --------------------------------------------------
def solve_vertical_east(los_a, los_d, le_a, lu_a, le_d, lu_d):

    los_d = los_d.interp(date=los_a.date)

    det = lu_a * le_d - le_a * lu_d
    if abs(det) < 1e-8:
        return None, None

    up = (le_d * los_a - le_a * los_d) / det
    east = (-lu_d * los_a + lu_a * los_d) / det

    return up, east


# --------------------------------------------------
# Extract vertical time series for all POIs
# --------------------------------------------------
vertical_series_dict = {}

for (idx_a, ra), (idx_d, rd) in zip(
        poi_geom_asc.iterrows(),
        poi_geom_desc.iterrows()):

    poi_name = ra["point"]   # <-- POI name

    xa, ya = ra["x"], ra["y"]
    xd, yd = rd["x"], rd["y"]

    los_a = los_asc.sel(x=xa, y=ya, method="nearest")
    los_d = los_desc.sel(x=xd, y=yd, method="nearest")

    up_ts, east_ts = solve_vertical_east(
        los_a, los_d,
        ra["look_E"], ra["look_U"],
        rd["look_E"], rd["look_U"]
    )

    if up_ts is None:
        continue

    s = up_ts.to_pandas()

    first_valid = s.first_valid_index()
    if first_valid is not None:
        s = s - s.loc[first_valid]

    vertical_series_dict[poi_name] = s


if not vertical_series_dict:
    raise RuntimeError("No vertical time series computed")


# --------------------------------------------------
# Align all POIs into one dataframe
# --------------------------------------------------
df = pd.DataFrame(vertical_series_dict)


# --------------------------------------------------
# Overall statistics
# --------------------------------------------------
mean_vertical = df.mean(axis=1, skipna=True)
median_vertical = df.median(axis=1, skipna=True)

min_vertical = df.min(axis=1, skipna=True)
max_vertical = df.max(axis=1, skipna=True)


# --------------------------------------------------
# Selected POIs by NAME
# --------------------------------------------------
selected_names = ["D1","D2","D3","D4","D5","D6","D7","D8","D9","D10","D11","D12","D13","D14","D15","D16","D17","D18","D19","D20","D21","D22","D23","D24","D25","D26","D27","D28"]
#["B1","B2","B3","B4","B5","B6","B7","B8","B9","B10","B11","B12","B13","B14","B15","B16","B17","B18","B19","B20","B21","B22","B23","B24"]

##["B1","B2","B3","B4","B5","B6","B7","B8","B9","B10","B11","B12","B13","B14","B15","B16","B17","B18","B19","B20","B21","B22","B23","B24"]
#["A1","A2","A3","A4","A5","A6","A7","A8","A9","A10","A11","A12","A13","A14","A15","A16","A17","A18","A19","A20","A21","A22","A23","A24","A25","A26","A27","A28","A29","A30","A31","A32","A33","A34","A35","A36","A37","A38","A39","A40","A41","A42","A43","A44","A45","A46"]   # <-- change to your POI names

df_selected = df[selected_names]

mean_selected = df_selected.mean(axis=1, skipna=True)
min_selected = df_selected.min(axis=1, skipna=True)
max_selected = df_selected.max(axis=1, skipna=True)


# --------------------------------------------------
# Plot
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

# Overall range
ax.fill_between(
    mean_vertical.index,
    min_vertical,
    max_vertical,
    color="lightgray",
    alpha=0.5,
    label="All POIs Range"
)

# Selected range
ax.fill_between(
    mean_selected.index,
    min_selected,
    max_selected,
    color="orange",
    alpha=0.3,
    label="Selected POIs 'A' Range"
)

# Overall mean
ax.plot(
    mean_vertical.index,
    mean_vertical,
    color="tab:red",
    lw=2.5,
    label="All POIs Mean"
)

# Selected mean
ax.plot(
    mean_selected.index,
    mean_selected,
    color="darkorange",
    lw=2.5,
    label="Selected POIs 'A' Mean"
)


ax.axhline(0, color="k", lw=0.8)

ax.set_xlabel("Date")
ax.set_ylabel("Vertical Displacement [mm]")
ax.set_title("Vertical Displacement Statistics\nAll POIs vs Selected POIs 'A' ")

ax.grid(True, linestyle="--", alpha=0.3)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.tick_params(axis="x", rotation=45)

ax.legend()

plt.tight_layout()

plt.savefig(
    "Airport_Vertical_Mean_Range_All_vs_Selected_POIs_B60_Y1619_PoA.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


# --------------------------------------------------
# Vertical + East solver
# --------------------------------------------------
def solve_vertical_east(los_a, los_d, le_a, lu_a, le_d, lu_d):

    los_d = los_d.interp(date=los_a.date)

    det = lu_a * le_d - le_a * lu_d
    if abs(det) < 1e-8:
        return None, None

    up = (le_d * los_a - le_a * los_d) / det
    east = (-lu_d * los_a + lu_a * los_d) / det

    return up, east


# --------------------------------------------------
# Extract vertical time series for all POIs
# --------------------------------------------------
vertical_series_dict = {}

for (idx_a, ra), (idx_d, rd) in zip(
        poi_geom_asc.iterrows(),
        poi_geom_desc.iterrows()):

    poi_name = ra["point"]   # <-- POI name

    xa, ya = ra["x"], ra["y"]
    xd, yd = rd["x"], rd["y"]

    los_a = los_asc.sel(x=xa, y=ya, method="nearest")
    los_d = los_desc.sel(x=xd, y=yd, method="nearest")

    up_ts, east_ts = solve_vertical_east(
        los_a, los_d,
        ra["look_E"], ra["look_U"],
        rd["look_E"], rd["look_U"]
    )

    if up_ts is None:
        continue

    s = up_ts.to_pandas()

    first_valid = s.first_valid_index()
    if first_valid is not None:
        s = s - s.loc[first_valid]

    vertical_series_dict[poi_name] = s


if not vertical_series_dict:
    raise RuntimeError("No vertical time series computed")


# --------------------------------------------------
# Align all POIs into one dataframe
# --------------------------------------------------
df = pd.DataFrame(vertical_series_dict)


# --------------------------------------------------
# Overall statistics
# --------------------------------------------------
mean_vertical = df.mean(axis=1, skipna=True)
median_vertical = df.median(axis=1, skipna=True)

min_vertical = df.min(axis=1, skipna=True)
max_vertical = df.max(axis=1, skipna=True)


# --------------------------------------------------
# Selected POIs by NAME
# --------------------------------------------------
selected_names = ["B1","B2","B3","B4","B5","B6","B7","B8","B9","B10","B11","B12","B13","B14","B15","B16","B17","B18","B19","B20","B21","B22","B23","B24"]

#["D1","D2","D3","D4","D5","D6","D7","D8","D9","D10","D11","D12","D13","D14","D15","D16","D17","D18","D19","D20","D21","D22","D23","D24","D25","D26","D27","D28"]
#["B1","B2","B3","B4","B5","B6","B7","B8","B9","B10","B11","B12","B13","B14","B15","B16","B17","B18","B19","B20","B21","B22","B23","B24"]

##["B1","B2","B3","B4","B5","B6","B7","B8","B9","B10","B11","B12","B13","B14","B15","B16","B17","B18","B19","B20","B21","B22","B23","B24"]
#["A1","A2","A3","A4","A5","A6","A7","A8","A9","A10","A11","A12","A13","A14","A15","A16","A17","A18","A19","A20","A21","A22","A23","A24","A25","A26","A27","A28","A29","A30","A31","A32","A33","A34","A35","A36","A37","A38","A39","A40","A41","A42","A43","A44","A45","A46"]   # <-- change to your POI names

df_selected = df[selected_names]

mean_selected = df_selected.mean(axis=1, skipna=True)
min_selected = df_selected.min(axis=1, skipna=True)
max_selected = df_selected.max(axis=1, skipna=True)


# --------------------------------------------------
# Plot
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

# Overall range
ax.fill_between(
    mean_vertical.index,
    min_vertical,
    max_vertical,
    color="lightgray",
    alpha=0.5,
    label="All POIs Range"
)

# Selected range
ax.fill_between(
    mean_selected.index,
    min_selected,
    max_selected,
    color="orange",
    alpha=0.3,
    label="Selected POIs 'B' Range"
)

# Overall mean
ax.plot(
    mean_vertical.index,
    mean_vertical,
    color="tab:red",
    lw=2.5,
    label="All POIs Mean"
)

# Selected mean
ax.plot(
    mean_selected.index,
    mean_selected,
    color="darkorange",
    lw=2.5,
    label="Selected POIs 'B' Mean"
)


ax.axhline(0, color="k", lw=0.8)

ax.set_xlabel("Date")
ax.set_ylabel("Vertical Displacement [mm]")
ax.set_title("Vertical Displacement Statistics\nAll POIs vs Selected POIs 'B' ")

ax.grid(True, linestyle="--", alpha=0.3)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.tick_params(axis="x", rotation=45)

ax.legend()

plt.tight_layout()

plt.savefig(
    "Airport_Vertical_Mean_Range_All_vs_Selected_POIs_B60_Y1619_PoB.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()



In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


# --------------------------------------------------
# Vertical + East solver
# --------------------------------------------------
def solve_vertical_east(los_a, los_d, le_a, lu_a, le_d, lu_d):

    los_d = los_d.interp(date=los_a.date)

    det = lu_a * le_d - le_a * lu_d
    if abs(det) < 1e-8:
        return None, None

    up = (le_d * los_a - le_a * los_d) / det
    east = (-lu_d * los_a + lu_a * los_d) / det

    return up, east


# --------------------------------------------------
# Extract vertical time series for all POIs
# --------------------------------------------------
vertical_series_dict = {}

for (idx_a, ra), (idx_d, rd) in zip(
        poi_geom_asc.iterrows(),
        poi_geom_desc.iterrows()):

    poi_name = ra["point"]   # <-- POI name

    xa, ya = ra["x"], ra["y"]
    xd, yd = rd["x"], rd["y"]

    los_a = los_asc.sel(x=xa, y=ya, method="nearest")
    los_d = los_desc.sel(x=xd, y=yd, method="nearest")

    up_ts, east_ts = solve_vertical_east(
        los_a, los_d,
        ra["look_E"], ra["look_U"],
        rd["look_E"], rd["look_U"]
    )

    if up_ts is None:
        continue

    s = up_ts.to_pandas()

    first_valid = s.first_valid_index()
    if first_valid is not None:
        s = s - s.loc[first_valid]

    vertical_series_dict[poi_name] = s


if not vertical_series_dict:
    raise RuntimeError("No vertical time series computed")


# --------------------------------------------------
# Align all POIs into one dataframe
# --------------------------------------------------
df = pd.DataFrame(vertical_series_dict)


# --------------------------------------------------
# Overall statistics
# --------------------------------------------------
mean_vertical = df.mean(axis=1, skipna=True)
median_vertical = df.median(axis=1, skipna=True)

min_vertical = df.min(axis=1, skipna=True)
max_vertical = df.max(axis=1, skipna=True)


# --------------------------------------------------
# Selected POIs by NAME
# --------------------------------------------------
selected_names =["B1","B2","B3","B4","B5","B6","B7","B8","B9","B10","B11","B12","B13","B14","B15","B16","B17","B18","B19","B20","B21","B22","B23","B24"]
 #["D1","D2","D3","D4","D5","D6","D7","D8","D9","D10","D11","D12","D13","D14","D15","D16","D17","D18","D19","D20","D21","D22","D23","D24","D25","D26","D27","D28"]

##["B1","B2","B3","B4","B5","B6","B7","B8","B9","B10","B11","B12","B13","B14","B15","B16","B17","B18","B19","B20","B21","B22","B23","B24"]
#["A1","A2","A3","A4","A5","A6","A7","A8","A9","A10","A11","A12","A13","A14","A15","A16","A17","A18","A19","A20","A21","A22","A23","A24","A25","A26","A27","A28","A29","A30","A31","A32","A33","A34","A35","A36","A37","A38","A39","A40","A41","A42","A43","A44","A45","A46"]   # <-- change to your POI names

df_selected = df[selected_names]

mean_selected = df_selected.mean(axis=1, skipna=True)
min_selected = df_selected.min(axis=1, skipna=True)
max_selected = df_selected.max(axis=1, skipna=True)


# --------------------------------------------------
# Plot
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

# Overall range
ax.fill_between(
    mean_vertical.index,
    min_vertical,
    max_vertical,
    color="lightgray",
    alpha=0.5,
    label="All POIs Range"
)

# Selected range
ax.fill_between(
    mean_selected.index,
    min_selected,
    max_selected,
    color="orange",
    alpha=0.3,
    label="Selected POIs 'B' Range"
)

# Overall mean
ax.plot(
    mean_vertical.index,
    mean_vertical,
    color="tab:red",
    lw=2.5,
    label="All POIs Mean"
)

# Selected mean
ax.plot(
    mean_selected.index,
    mean_selected,
    color="darkorange",
    lw=2.5,
    label="Selected POIs 'B' Mean"
)


ax.axhline(0, color="k", lw=0.8)

ax.set_xlabel("Date")
ax.set_ylabel("Vertical Displacement [mm]")
ax.set_title("Vertical Displacement Statistics\nAll POIs vs Selected POIs 'B' ")

ax.grid(True, linestyle="--", alpha=0.3)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.tick_params(axis="x", rotation=45)

ax.legend()

plt.tight_layout()
plt.show()

plt.savefig(
    "Airport_Vertical_Mean_Range_All_vs_Selected_POIs_B60_Y1619_PoB.png",
    dpi=300,
    bbox_inches="tight"
)



In [ ]:
# --------------------------------------------------
# Selected POIs by NAME
# --------------------------------------------------
selected_names_A = [
    "D1","D2","D3","D4","D5","D6","D7","D8","D9","D10",
    "D11","D12","D13","D14","D15","D16","D17","D18","D19","D20",
    "D21","D22","D23","D24","D25","D26","D27","D28"
]

selected_names_B = [
    "B1","B2","B3","B4","B5","B6","B7","B8","B9","B10",
    "B11","B12","B13","B14","B15","B16","B17","B18","B19","B20",
    "B21","B22","B23","B24"
]

# Keep only names that exist in df
selected_names_A = [p for p in selected_names_A if p in df.columns]
selected_names_B = [p for p in selected_names_B if p in df.columns]

if not selected_names_A:
    raise RuntimeError("No selected POIs found for group A")

if not selected_names_B:
    raise RuntimeError("No selected POIs found for group B")

df_selected_A = df[selected_names_A]
df_selected_B = df[selected_names_B]

mean_selected_A = df_selected_A.mean(axis=1, skipna=True)
min_selected_A = df_selected_A.min(axis=1, skipna=True)
max_selected_A = df_selected_A.max(axis=1, skipna=True)

mean_selected_B = df_selected_B.mean(axis=1, skipna=True)
min_selected_B = df_selected_B.min(axis=1, skipna=True)
max_selected_B = df_selected_B.max(axis=1, skipna=True)


# --------------------------------------------------
# Plot
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

# Selected POIs A range
ax.fill_between(
    mean_selected_A.index,
    min_selected_A,
    max_selected_A,
    color="orange",
    alpha=0.25,
    label="Selected POIs 'A' Range"
)

# Selected POIs B range
ax.fill_between(
    mean_selected_B.index,
    min_selected_B,
    max_selected_B,
    color="tab:blue",
    alpha=0.20,
    label="Selected POIs 'B' Range"
)

# Selected POIs A mean
ax.plot(
    mean_selected_A.index,
    mean_selected_A,
    color="darkorange",
    lw=2.5,
    label="Selected POIs 'A' Mean"
)

# Selected POIs B mean
ax.plot(
    mean_selected_B.index,
    mean_selected_B,
    color="tab:blue",
    lw=2.5,
    label="Selected POIs 'B' Mean"
)

ax.axhline(0, color="k", lw=0.8)

ax.set_xlabel("Date")
ax.set_ylabel("Vertical Displacement [mm]")
ax.set_title("Vertical Displacement Statistics\nSelected POIs 'A' vs Selected POIs 'B'")

ax.grid(True, linestyle="--", alpha=0.3)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.tick_params(axis="x", rotation=45)

ax.legend()

plt.tight_layout()

plt.savefig(
    "Airport_Vertical_Mean_Range_Selected_POIs_A_vs_B.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


# --------------------------------------------------
# Vertical + East solver
# --------------------------------------------------
def solve_vertical_east(los_a, los_d, le_a, lu_a, le_d, lu_d):

    los_d = los_d.interp(date=los_a.date)

    det = lu_a * le_d - le_a * lu_d
    if abs(det) < 1e-8:
        return None, None

    up = (le_d * los_a - le_a * los_d) / det
    east = (-lu_d * los_a + lu_a * los_d) / det

    return up, east


# --------------------------------------------------
# Extract EAST time series for all POIs
# --------------------------------------------------
east_series_dict = {}

for (idx_a, ra), (idx_d, rd) in zip(
        poi_geom_asc.iterrows(),
        poi_geom_desc.iterrows()):

    poi_name = ra["point"]

    xa, ya = ra["x"], ra["y"]
    xd, yd = rd["x"], rd["y"]

    los_a = los_asc.sel(x=xa, y=ya, method="nearest")
    los_d = los_desc.sel(x=xd, y=yd, method="nearest")

    up_ts, east_ts = solve_vertical_east(
        los_a, los_d,
        ra["look_E"], ra["look_U"],
        rd["look_E"], rd["look_U"]
    )

    if east_ts is None:
        continue

    s = east_ts.to_pandas()

    first_valid = s.first_valid_index()
    if first_valid is not None:
        s = s - s.loc[first_valid]

    east_series_dict[poi_name] = s


if not east_series_dict:
    raise RuntimeError("No east-west time series computed")


# --------------------------------------------------
# Align into dataframe
# --------------------------------------------------
df = pd.DataFrame(east_series_dict)


# --------------------------------------------------
# Overall statistics
# --------------------------------------------------
mean_east = df.mean(axis=1, skipna=True)

min_east = df.min(axis=1, skipna=True)
max_east = df.max(axis=1, skipna=True)


# --------------------------------------------------
# Selected POIs by name
# --------------------------------------------------

df_selected = df[selected_names]

mean_selected = df_selected.mean(axis=1, skipna=True)
min_selected = df_selected.min(axis=1, skipna=True)
max_selected = df_selected.max(axis=1, skipna=True)


# --------------------------------------------------
# Plot
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

# Overall range
ax.fill_between(
    mean_east.index,
    min_east,
    max_east,
    color="lightgray",
    alpha=0.5,
    label="All POIs Range"
)

# Selected range
ax.fill_between(
    mean_selected.index,
    min_selected,
    max_selected,
    color="orange",
    alpha=0.3,
    label="Selected POIs Range"
)

# Overall mean
ax.plot(
    mean_east.index,
    mean_east,
    color="tab:green",
    lw=2.5,
    label="All POIs Mean"
)

# Selected mean
ax.plot(
    mean_selected.index,
    mean_selected,
    color="darkorange",
    lw=2.5,
    label="Selected POIs Mean"
)

ax.axhline(0, color="k", lw=0.8)

ax.set_xlabel("Date")
ax.set_ylabel("East-West Displacement [mm]")
ax.set_title("East-West Displacement Statistics\nAll POIs vs Selected POIs")

ax.grid(True, linestyle="--", alpha=0.3)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.tick_params(axis="x", rotation=45)

ax.legend()

plt.tight_layout()

plt.savefig(
    "Island1_EastWest_Mean_Range_All_vs_Selected_POIs_B60_Y18_25.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# --------------------------------------------------
# Load LOS displacement cubes
# --------------------------------------------------


# --------------------------------------------------
# Correct vertical solver
# --------------------------------------------------
def solve_vertical(los_a, los_d, le_a, lu_a, le_d, lu_d):

    los_d = los_d.interp(date=los_a.date)

    det = lu_a * le_d - le_a * lu_d
    if abs(det) < 1e-8:
        return None

    up = (le_d * los_a - le_a * los_d) / det
    return up.astype("float64")

# --------------------------------------------------
# Extract full-resolution vertical time series
# --------------------------------------------------
vertical_dict = {}
xy_dict = {}

for (_, ra), (_, rd) in zip(poi_geom_asc.iterrows(), poi_geom_desc.iterrows()):

    name = ra["point"]

    xa, ya = ra["x"], ra["y"]
    xd, yd = rd["x"], rd["y"]

    los_a = los_asc.sel(x=xa, y=ya, method="nearest").squeeze()
    los_d = los_desc.sel(x=xd, y=yd, method="nearest").squeeze()

    vertical_ts = solve_vertical(
        los_a, los_d,
        ra["look_E"], ra["look_U"],
        rd["look_E"], rd["look_U"]
    )

    if vertical_ts is not None:
        vertical_dict[name] = vertical_ts
        xy_dict[name] = (xa, ya)

if not vertical_dict:
    raise RuntimeError("No vertical data computed")

# --------------------------------------------------
# Zero reference to first valid epoch
# --------------------------------------------------
vertical_rel_dict = {}

for name, ts in vertical_dict.items():
    s = ts.to_pandas()
    first_valid = s.first_valid_index()

    if first_valid is not None:
        vertical_rel_dict[name] = s - s.loc[first_valid]

# --------------------------------------------------
# Use full time resolution
# --------------------------------------------------
dates = vertical_rel_dict[next(iter(vertical_rel_dict))].index

# Robust magnitude scaling
all_vals = np.concatenate([
    np.abs(ts.dropna().values)
    for ts in vertical_rel_dict.values()
])

ref_mag = np.percentile(all_vals, 90)

# --------------------------------------------------
# Animation setup
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 6), dpi=150)

VIS_SCALE = 1.0
MIN_LEN = 0.2
MAX_LEN = 1.5
QUIVER_SCALE = 10
FRAME_INTERVAL = 800   # faster since more frames
# --------------------------------------------------
# Define reference magnitude for scale bar (physical units)
# --------------------------------------------------
scale_value = ref_mag   # you can set manually, e.g. 5 or 10 mm
scale_arrow = np.clip(scale_value / ref_mag, MIN_LEN, MAX_LEN) * VIS_SCALE

# --------------------------------------------------
# Animation update
# --------------------------------------------------
def update(frame):

    ax.clear()
    date = dates[frame]

    Xu, Yu, Uu, Vu = [], [], [], []
    Xs, Ys, Us, Vs = [], [], [], []

    for name, ts in vertical_rel_dict.items():

        if date not in ts.index:
            continue

        val = ts.loc[date]
        if not np.isfinite(val):
            continue

        x, y = xy_dict[name]
        mag = np.clip(abs(val) / ref_mag, MIN_LEN, MAX_LEN)

        if val >= 0:
            Xu.append(x); Yu.append(y)
            Uu.append(0.0); Vu.append(+mag * VIS_SCALE)
        else:
            Xs.append(x); Ys.append(y)
            Us.append(0.0); Vs.append(-mag * VIS_SCALE)

    ax.scatter(Xu + Xs, Yu + Ys, s=6, color="k", alpha=0.4, zorder=1)

    q = None

    if Xu:
        q = ax.quiver(
            Xu, Yu, Uu, Vu,
            angles="xy",
            scale_units="width",
            scale=QUIVER_SCALE,
            width=0.005,
            pivot="middle",
            color="red",
            zorder=3
        )

    if Xs:
        q = ax.quiver(
            Xs, Ys, Us, Vs,
            angles="xy",
            scale_units="width",
            scale=QUIVER_SCALE,
            width=0.005,
            pivot="middle",
            color="blue",
            zorder=3
        )

    # --------------------------------------------------
    # Add displacement magnitude scale bar
    # --------------------------------------------------
    if q is not None:
        ax.quiverkey(
            q,
            X=0.85, Y=0.92,
            U=scale_arrow,
            label=f"{scale_value:.2f} mm",
            labelpos="E",
            coordinates="axes"
        )

    ax.set_aspect("auto")
    ax.set_title(f"Vertical Displacement  {date.strftime('%Y-%m-%d')}", fontsize=14)
    ax.set_xlabel("X coordinate")
    ax.set_ylabel("Y coordinate")
    ax.grid(True, linestyle="--", alpha=0.3)

# --------------------------------------------------
# Create animation
# --------------------------------------------------
anim = FuncAnimation(
    fig,
    update,
    frames=len(dates),
    interval=FRAME_INTERVAL,
    repeat=True
)

gif_path = "SBAS_Airport_Vertical_AllEpochs_Y16_19_B90_withScale_Feb26.gif"

anim.save(
    gif_path,
    writer=PillowWriter(fps=1.5)
)

plt.close(fig)

print("Animated GIF saved to:", gif_path)


In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# --------------------------------------------------
# Load LOS displacement cubes
# --------------------------------------------------

# --------------------------------------------------
# Correct vertical + east solver
# --------------------------------------------------
def solve_vertical_east(los_a, los_d, le_a, lu_a, le_d, lu_d):

    los_d = los_d.interp(date=los_a.date)

    det = lu_a * le_d - le_a * lu_d
    if abs(det) < 1e-8:
        return None, None

    up = (le_d * los_a - le_a * los_d) / det
    east = (-lu_d * los_a + lu_a * los_d) / det

    return up.astype("float64"), east.astype("float64")

# --------------------------------------------------
# Extract full-resolution East time series
# --------------------------------------------------
east_dict = {}
xy_dict = {}

for (_, ra), (_, rd) in zip(poi_geom_asc.iterrows(), poi_geom_desc.iterrows()):

    name = ra["point"]

    xa, ya = ra["x"], ra["y"]
    xd, yd = rd["x"], rd["y"]

    los_a = los_asc.sel(x=xa, y=ya, method="nearest").squeeze()
    los_d = los_desc.sel(x=xd, y=yd, method="nearest").squeeze()

    up_ts, east_ts = solve_vertical_east(
        los_a, los_d,
        ra["look_E"], ra["look_U"],
        rd["look_E"], rd["look_U"]
    )

    if east_ts is not None:
        east_dict[name] = east_ts
        xy_dict[name] = (xa, ya)

if not east_dict:
    raise RuntimeError("No East-West data computed")

# --------------------------------------------------
# Zero reference to first valid epoch
# --------------------------------------------------
east_rel_dict = {}

for name, ts in east_dict.items():
    s = ts.to_pandas()
    first_valid = s.first_valid_index()

    if first_valid is not None:
        east_rel_dict[name] = s - s.loc[first_valid]

# --------------------------------------------------
# Use full time resolution
# --------------------------------------------------
dates = east_rel_dict[next(iter(east_rel_dict))].index

# Robust magnitude scaling
all_vals = np.concatenate([
    np.abs(ts.dropna().values)
    for ts in east_rel_dict.values()
])

ref_mag = np.percentile(all_vals, 90)

# --------------------------------------------------
# Animation setup
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 6), dpi=150)

VIS_SCALE = 1.0
MIN_LEN = 0.2
MAX_LEN = 1.5
QUIVER_SCALE = 10
FRAME_INTERVAL = 800
# --------------------------------------------------
# Define reference magnitude for scale bar
# --------------------------------------------------
scale_value = ref_mag          # or set manually, e.g. 5 or 10 mm
scale_arrow = np.clip(scale_value / ref_mag, MIN_LEN, MAX_LEN) * VIS_SCALE

# --------------------------------------------------
# Animation update
# --------------------------------------------------
def update(frame):

    ax.clear()
    date = dates[frame]

    Xe, Ye, Ue, Ve = [], [], [], []
    Xw, Yw, Uw, Vw = [], [], [], []

    for name, ts in east_rel_dict.items():

        if date not in ts.index:
            continue

        val = ts.loc[date]
        if not np.isfinite(val):
            continue

        x, y = xy_dict[name]
        mag = np.clip(abs(val) / ref_mag, MIN_LEN, MAX_LEN)

        if val >= 0:
            Xe.append(x); Ye.append(y)
            Ue.append(+mag * VIS_SCALE); Ve.append(0.0)
        else:
            Xw.append(x); Yw.append(y)
            Uw.append(-mag * VIS_SCALE); Vw.append(0.0)

    ax.scatter(Xe + Xw, Ye + Yw, s=6, color="k", alpha=0.4, zorder=1)

    q = None

    if Xe:
        q = ax.quiver(
            Xe, Ye, Ue, Ve,
            angles="xy",
            scale_units="width",
            scale=QUIVER_SCALE,
            width=0.005,
            pivot="middle",
            color="red",
            zorder=3
        )

    if Xw:
        q = ax.quiver(
            Xw, Yw, Uw, Vw,
            angles="xy",
            scale_units="width",
            scale=QUIVER_SCALE,
            width=0.005,
            pivot="middle",
            color="blue",
            zorder=3
        )

    # --------------------------------------------------
    # Add displacement magnitude reference scale
    # --------------------------------------------------
    if q is not None:
        ax.quiverkey(
            q,
            X=0.85, Y=0.92,
            U=scale_arrow,
            label=f"{scale_value:.2f} mm",
            labelpos="E",
            coordinates="axes"
        )

    ax.set_aspect("auto")
    ax.set_title(f"East-West Displacement  {date.strftime('%Y-%m-%d')}", fontsize=14)
    ax.set_xlabel("X coordinate")
    ax.set_ylabel("Y coordinate")
    ax.grid(True, linestyle="--", alpha=0.3)


# --------------------------------------------------
# Create animation
# --------------------------------------------------
anim = FuncAnimation(
    fig,
    update,
    frames=len(dates),
    interval=FRAME_INTERVAL,
    repeat=True
)

gif_path = "SBAS_Airport_EastWest_AllEpochs_Y16_19_B90_withScale_Feb26.gif"

anim.save(
    gif_path,
    writer=PillowWriter(fps=1.5)
)

plt.close(fig)

print("Animated GIF saved to:", gif_path)


In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# --------------------------------------------------
# Vertical + East solver
# --------------------------------------------------
def solve_vertical_east(los_a, los_d, le_a, lu_a, le_d, lu_d):

    los_d = los_d.interp(date=los_a.date)

    det = lu_a * le_d - le_a * lu_d
    if abs(det) < 1e-8:
        return None, None

    up = (le_d * los_a - le_a * los_d) / det
    east = (-lu_d * los_a + lu_a * los_d) / det

    return up, east

# --------------------------------------------------
# Extract vertical time series for all POIs
# --------------------------------------------------
vertical_series_list = []

for (_, ra), (_, rd) in zip(
        poi_geom_asc.iterrows(),
        poi_geom_desc.iterrows()):

    xa, ya = ra["x"], ra["y"]
    xd, yd = rd["x"], rd["y"]

    los_a = los_asc.sel(x=xa, y=ya, method="nearest")
    los_d = los_desc.sel(x=xd, y=yd, method="nearest")

    up_ts, east_ts = solve_vertical_east(
        los_a, los_d,
        ra["look_E"], ra["look_U"],
        rd["look_E"], rd["look_U"]
    )

    if up_ts is None:
        continue

    s = up_ts.to_pandas()

    first_valid = s.first_valid_index()
    if first_valid is not None:
        s = s - s.loc[first_valid]

    vertical_series_list.append(s)

if not vertical_series_list:
    raise RuntimeError("No vertical time series computed")

# --------------------------------------------------
# Align all POIs into one dataframe
# --------------------------------------------------
df = pd.concat(vertical_series_list, axis=1)

# Mean vertical displacement per epoch
mean_vertical = df.mean(axis=1, skipna=True)
# Median vertical displacement per epoch
median_vertical = df.median(axis=1, skipna=True)

# Range per epoch
min_vertical = df.min(axis=1, skipna=True)
max_vertical = df.max(axis=1, skipna=True)
range_vertical = max_vertical - min_vertical

# --------------------------------------------------
# Plot mean + median + range
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

# Shaded range
ax.fill_between(
    mean_vertical.index,
    min_vertical,
    max_vertical,
    color="lightgray",
    alpha=0.5,
    label="Range (Min to Max)"
)

# Mean line
ax.plot(
    mean_vertical.index,
    mean_vertical,
    color="tab:red",
    lw=2.5,
    label="Mean Vertical"
)

# Median line
ax.plot(
    median_vertical.index,
    median_vertical,
    color="tab:blue",
    lw=2.0,
    linestyle="--",
    label="Median Vertical"
)

ax.axhline(0, color="k", lw=0.8)

ax.set_xlabel("Date")
ax.set_ylabel("Vertical Displacement [mm]")
ax.set_title("Mean, Median and Range of Vertical Displacement\nAll POIs")

ax.grid(True, linestyle="--", alpha=0.3)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.tick_params(axis="x", rotation=45)

ax.legend()

plt.tight_layout()

plt.savefig(
    "Airport_Vertical_Mean_Median_Range_All_POIs_B90_Y16_19_SBAS.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()



In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# --------------------------------------------------
# Vertical + East solver
# --------------------------------------------------
def solve_vertical_east(los_a, los_d, le_a, lu_a, le_d, lu_d):

    los_d = los_d.interp(date=los_a.date)

    det = lu_a * le_d - le_a * lu_d
    if abs(det) < 1e-8:
        return None, None

    up = (le_d * los_a - le_a * los_d) / det
    east = (-lu_d * los_a + lu_a * los_d) / det

    return up, east

# --------------------------------------------------
# Extract East-West time series for all POIs
# --------------------------------------------------
east_series_list = []

for (_, ra), (_, rd) in zip(
        poi_geom_asc.iterrows(),
        poi_geom_desc.iterrows()):

    xa, ya = ra["x"], ra["y"]
    xd, yd = rd["x"], rd["y"]

    los_a = los_asc.sel(x=xa, y=ya, method="nearest")
    los_d = los_desc.sel(x=xd, y=yd, method="nearest")

    up_ts, east_ts = solve_vertical_east(
        los_a, los_d,
        ra["look_E"], ra["look_U"],
        rd["look_E"], rd["look_U"]
    )

    if east_ts is None:
        continue

    s = east_ts.to_pandas()

    first_valid = s.first_valid_index()
    if first_valid is not None:
        s = s - s.loc[first_valid]

    east_series_list.append(s)

if not east_series_list:
    raise RuntimeError("No East-West time series computed")

# --------------------------------------------------
# Align all POIs into one dataframe
# --------------------------------------------------
df = pd.concat(east_series_list, axis=1)

# Mean East-West displacement per epoch
mean_east = df.mean(axis=1, skipna=True)

# Range per epoch
min_east = df.min(axis=1, skipna=True)
max_east = df.max(axis=1, skipna=True)
range_east = max_east - min_east

# --------------------------------------------------
# Plot mean + range
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

# Shaded range
ax.fill_between(
    mean_east.index,
    min_east,
    max_east,
    color="lightgray",
    alpha=0.5,
    label="Range (Min to Max)"
)

# Mean line
ax.plot(
    mean_east.index,
    mean_east,
    color="tab:purple",
    lw=2.5,
    label="Mean East-West"
)

ax.axhline(0, color="k", lw=0.8)

ax.set_xlabel("Date")
ax.set_ylabel("East-West Displacement [mm]")
ax.set_title("Mean and Range of East-West Displacement\nAll POIs")

ax.grid(True, linestyle="--", alpha=0.3)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.tick_params(axis="x", rotation=45)

ax.legend()

plt.tight_layout()

plt.savefig(
    "Island_1EastWest_Mean_and_Range_All_POIs_B60_Y18_25_SBAS.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# --------------------------------------------------
# Load LOS displacement cubes
# --------------------------------------------------
#los_asc  = xr.open_dataarray("/scratch/pm4167/ADNOC/disp_sbas_ADIsland2_2021_24_Asc_B36_New3.nc")
#los_desc = xr.open_dataarray("/scratch/pm4167/ADNOC/disp_sbas_ADIsland2_2021_24_Desc_B36_New4.nc")

# --------------------------------------------------
# Load POI geometry with look vectors
# --------------------------------------------------
#poi_geom_asc  = pd.read_csv("/scratch/pm4167/ADNOC/poi_geometryALLPoints_angles_AD_island2_2021_24_ASC_B48_New4_1.csv")
#poi_geom_desc = pd.read_csv("/scratch/pm4167/ADNOC/poi_geometryALLPoints_angles_AD_island2_2021_24_DESC_B36_New4.csv")

# --------------------------------------------------
# Vertical + East solver
# --------------------------------------------------
def solve_vertical_east(los_a, los_d, le_a, lu_a, le_d, lu_d):
    los_d = los_d.interp(date=los_a.date)

    det = lu_a * le_d - le_a * lu_d
    if abs(det) < 1e-8:
        return None, None

    up = ( le_d * los_a - le_a * los_d ) / det
    east = (-lu_d * los_a + lu_a * los_d ) / det

    return up, east



# --------------------------------------------------
# Figure layout
# --------------------------------------------------
n = len(poi_geom_asc)
ncols = 5
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(18, 4 * nrows),
    sharex=True,
    dpi=300
)

axes = axes.flatten()

# --------------------------------------------------
# Loop over POIs
# --------------------------------------------------
for ax, (_, ra), (_, rd) in zip(
        axes,
        poi_geom_asc.iterrows(),
        poi_geom_desc.iterrows()):

    name = ra["point"]

    xa, ya = ra["x"], ra["y"]
    xd, yd = rd["x"], rd["y"]

    los_a = los_asc.sel(x=xa, y=ya, method="nearest")
    los_d = los_desc.sel(x=xd, y=yd, method="nearest")

    le_a, lu_a = ra["look_E"], ra["look_U"]
    le_d, lu_d = rd["look_E"], rd["look_U"]

    up_ts, east_ts = solve_vertical_east(
        los_a, los_d,
        le_a, lu_a,
        le_d, lu_d
    )

    # ---------------- Plot LOS ----------------
    ax.plot(
        los_a.date, los_a,
        lw=1.4, color="tab:blue",
        label="Asc LOS"
    )

    ax.plot(
        los_d.date, los_d,
        lw=1.4, color="tab:green",
        label="Desc LOS"
    )

    # ---------------- Plot decomposed components ----------------
    if up_ts is not None:
        ax.plot(
            up_ts.date, up_ts,
            lw=2.0, ls="--", color="tab:red",
            label="Vertical"
        )

    if east_ts is not None:
        ax.plot(
            east_ts.date, east_ts,
            lw=2.0, ls=":", color="tab:purple",
            label="East West"
        )

    ax.axhline(0, color="k", lw=0.6)
    ax.set_title(name, fontsize=11)
    ax.grid(True, linestyle="--", alpha=0.3)

    dates = los_a.date.values
    tick_idx = np.linspace(0, len(dates) - 1, 10, dtype=int)
    ax.set_xticks(dates[tick_idx])
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.tick_params(axis="x", rotation=45)

# --------------------------------------------------
# Remove unused axes
# --------------------------------------------------
for i in range(len(poi_geom_asc), len(axes)):
    fig.delaxes(axes[i])

# --------------------------------------------------
# Shared labels and legend
# --------------------------------------------------
fig.text(0.5, 0.04, "Date", ha="center", fontsize=12)
fig.text(0.04, 0.5, "Displacement [mm]",
         va="center", rotation="vertical", fontsize=12)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, fontsize=11)

fig.suptitle(
    "SBAS Time Series per POI\nAscending LOS + Descending LOS + Vertical + East West\nIsland 1 Baseline 60",
    fontsize=16
)

plt.tight_layout(rect=[0.03, 0.06, 1, 0.93])

plt.savefig(
    "SBAS_Displacement_AscDesc_Vert_EW_Subplots_AD_island1_2018_2025_B60_New4Plot_13Feb_col5.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
import math
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# --------------------------------------------------
# Load LOS displacement cubes
# --------------------------------------------------
#los_asc  = xr.open_dataarray("/scratch/pm4167/ADNOC/disp_ps_ADIsland2_2021_24_Asc_B36_New3.nc")
#los_desc = xr.open_dataarray("/scratch/pm4167/ADNOC/disp_ps_ADIsland2_2021_24_Desc_B36_New4.nc")

# --------------------------------------------------
# Load POI geometry with look vectors
# --------------------------------------------------
#poi_geom_asc  = pd.read_csv("/scratch/pm4167/ADNOC/poi_geometryALLPoints_angles_AD_island2_2021_24_ASC_B48_New4_1.csv")
#poi_geom_desc = pd.read_csv("/scratch/pm4167/ADNOC/poi_geometryALLPoints_angles_AD_island2_2021_24_DESC_B36_New4.csv")

# --------------------------------------------------
# Vertical + East solver
# --------------------------------------------------
def solve_vertical_east(los_a, los_d, le_a, lu_a, le_d, lu_d):
    los_d = los_d.interp(date=los_a.date)
    det = lu_a * le_d - le_a * lu_d
    if abs(det) < 1e-8:
        return None
    return ((-lu_d * los_a + lu_a * los_d) / det).astype("float64")

# --------------------------------------------------
# Extract EW time series
# --------------------------------------------------
ew_dict = {}
xy_dict = {}

for (_, ra), (_, rd) in zip(poi_geom_asc.iterrows(), poi_geom_desc.iterrows()):
    name = ra["point"]
    xa, ya = ra["x"], ra["y"]
    xd, yd = rd["x"], rd["y"]

    los_a = los_asc.sel(x=xa, y=ya, method="nearest").squeeze()
    los_d = los_desc.sel(x=xd, y=yd, method="nearest").squeeze()

    east_ts = solve_vertical_east(
        los_a, los_d,
        ra["look_E"], ra["look_U"],
        rd["look_E"], rd["look_U"]
    )

    if east_ts is not None:
        ew_dict[name] = east_ts
        xy_dict[name] = (xa, ya)

# --------------------------------------------------
# Zero reference to first valid epoch
# --------------------------------------------------
ew_rel_dict = {}

for name, ts in ew_dict.items():
    s = ts.to_pandas()
    first_valid = s.first_valid_index()
    if first_valid is not None:
        ew_rel_dict[name] = s - s.loc[first_valid]

# --------------------------------------------------
# Monthly averaging
# --------------------------------------------------
ew_monthly_dict = {}

for name, s in ew_rel_dict.items():
    m = s.resample("M").sum()

    if not m.isna().all():
        ew_monthly_dict[name] = m


if not ew_monthly_dict:
    raise RuntimeError("No monthly EW data available")

# --------------------------------------------------
# Robust reference magnitude for scaling
# --------------------------------------------------
all_vals = np.concatenate([
    np.abs(ts.dropna().values)
    for ts in ew_monthly_dict.values()
])

ref_mag = np.percentile(all_vals, 90)

# --------------------------------------------------
# Animation setup
# --------------------------------------------------
dates = ew_monthly_dict[next(iter(ew_monthly_dict))].index

fig, ax = plt.subplots(figsize=(6, 6), dpi=150)

VIS_SCALE = 1.0
MIN_LEN = 0.2
MAX_LEN = 1.5
QUIVER_SCALE = 10     # requested scale
FRAME_INTERVAL = 1200 # milliseconds (slow animation)

# --------------------------------------------------
# Animation update function
# --------------------------------------------------
def update(frame):
    ax.clear()
    date = dates[frame]

    Xe, Ye, Ue, Ve = [], [], [], []
    Xw, Yw, Uw, Vw = [], [], [], []

    for name, ts in ew_monthly_dict.items():
        if date not in ts.index:
            continue

        ew_val = ts.loc[date]
        if not np.isfinite(ew_val):
            continue

        x, y = xy_dict[name]
        mag = np.clip(abs(ew_val) / ref_mag, MIN_LEN, MAX_LEN)

        if ew_val >= 0:
            Xe.append(x); Ye.append(y)
            Ue.append(+mag * VIS_SCALE); Ve.append(0.0)
        else:
            Xw.append(x); Yw.append(y)
            Uw.append(-mag * VIS_SCALE); Vw.append(0.0)

    ax.scatter(
        Xe + Xw, Ye + Yw,
        s=6, color="k", alpha=0.4, zorder=1
    )

    if Xe:
        ax.quiver(
            Xe, Ye, Ue, Ve,
            angles="xy",
            scale_units="width",
            scale=QUIVER_SCALE,
            width=0.005,
            headwidth=6,
            headlength=8,
            headaxislength=6,
            pivot="middle",
            color="red",
            zorder=3
        )

    if Xw:
        ax.quiver(
            Xw, Yw, Uw, Vw,
            angles="xy",
            scale_units="width",
            scale=QUIVER_SCALE,
            width=0.005,
            headwidth=6,
            headlength=8,
            headaxislength=6,
            pivot="middle",
            color="blue",
            zorder=3
        )

    ax.set_aspect("auto")
    ax.set_title(f"East–West Displacement  {date.strftime('%Y-%m')}", fontsize=14)
    ax.set_xlabel("X coordinate")
    ax.set_ylabel("Y coordinate")
    ax.grid(True, linestyle="--", alpha=0.3)

# --------------------------------------------------
# Create and save animation
# --------------------------------------------------
anim = FuncAnimation(
    fig,
    update,
    frames=len(dates),
    interval=FRAME_INTERVAL,
    repeat=True
)

gif_path = "SBAS_Island5_EW_Displacement_Animation_Slow_Scale10_BL36_Y21_25cumMsum_13Feb.gif"

anim.save(
    gif_path,
    writer=PillowWriter(fps=0.8)
)

plt.close(fig)

print(f"Animated GIF saved to: {gif_path}")


In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# --------------------------------------------------
# Load LOS displacement cubes
# --------------------------------------------------
#los_asc  = xr.open_dataarray("/scratch/pm4167/ADNOC/disp_ps_ADIsland2_2021_24_Asc_B36_New3.nc")
#los_desc = xr.open_dataarray("/scratch/pm4167/ADNOC/disp_ps_ADIsland2_2021_24_Desc_B36_New4.nc")

# --------------------------------------------------
# Load POI geometry with look vectors
# --------------------------------------------------
#poi_geom_asc  = pd.read_csv("/scratch/pm4167/ADNOC/poi_geometryALLPoints_angles_AD_island2_2021_24_ASC_B48_New4_1.csv")
#poi_geom_desc = pd.read_csv("/scratch/pm4167/ADNOC/poi_geometryALLPoints_angles_AD_island2_2021_24_DESC_B36_New4.csv")



# --------------------------------------------------
# Select POIs of interest
# --------------------------------------------------
# --------------------------------------------------
# Select POIs of interest (optional)
# --------------------------------------------------
selected_points = []#"P3", "P4", "P5", "P6", "P7", "P8"
# selected_points = None  # uncomment to use all points

if len(selected_points) !=0:
    n = len(selected_points)
    poi_geom_asc = poi_geom_asc[poi_geom_asc["point"].isin(selected_points)]
    poi_geom_desc = poi_geom_desc[poi_geom_desc["point"].isin(selected_points)]

    # Ensure same ordering
    poi_geom_asc = poi_geom_asc.set_index("point").loc[selected_points].reset_index()
    poi_geom_desc = poi_geom_desc.set_index("point").loc[selected_points].reset_index()
else:
    # Use all points, enforce common set and ordering
    n = len(poi_geom_asc)
    common_points = np.intersect1d(
        poi_geom_asc["point"].values,
        poi_geom_desc["point"].values
    )

    poi_geom_asc = (
        poi_geom_asc.set_index("point")
        .loc[common_points]
        .reset_index()
    )
    poi_geom_desc = (
        poi_geom_desc.set_index("point")
        .loc[common_points]
        .reset_index()
    )


# --------------------------------------------------
# Vertical + East solver (EW only used)
# --------------------------------------------------
def solve_east(los_a, los_d, le_a, lu_a, le_d, lu_d):
    los_d = los_d.interp(date=los_a.date)

    det = lu_a * le_d - le_a * lu_d
    if abs(det) < 1e-8:
        return None

    east = (-lu_d * los_a + lu_a * los_d) / det
    return east

# --------------------------------------------------
# Figure layout
# --------------------------------------------------
#n = len(selected_points)
ncols = 5
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(16, 4 * nrows),
    sharex=True,
    dpi=300
)

axes = axes.flatten()

# --------------------------------------------------
# Loop over selected POIs
# --------------------------------------------------
for ax, (_, ra), (_, rd) in zip(
    axes,
    poi_geom_asc.iterrows(),
    poi_geom_desc.iterrows()
):

    name = ra["point"]

    xa, ya = ra["x"], ra["y"]
    xd, yd = rd["x"], rd["y"]

    los_a = los_asc.sel(x=xa, y=ya, method="nearest").squeeze()
    los_d = los_desc.sel(x=xd, y=yd, method="nearest").squeeze()

    east_ts = solve_east(
        los_a, los_d,
        ra["look_E"], ra["look_U"],
        rd["look_E"], rd["look_U"]
    )

    if east_ts is None:
        ax.set_title(f"{name} (invalid geometry)")
        continue

    # Zero-reference to first valid epoch
    s = east_ts.to_pandas()
    first_valid = s.first_valid_index()
    if first_valid is not None:
        s = s - s.loc[first_valid]

    # Plot EW component
    ax.plot(
        s.index, s.values,
        lw=2.2, color="tab:purple",
        label="East–West"
    )

    ax.axhline(0, color="k", lw=0.8)
    ax.set_title(name, fontsize=12)
    ax.grid(True, linestyle="--", alpha=0.3)

    # Time axis formatting
    tick_idx = np.linspace(0, len(s.index) - 1, 8, dtype=int)
    ax.set_xticks(s.index[tick_idx])
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.tick_params(axis="x", rotation=45)

# --------------------------------------------------
# Remove unused axes
# --------------------------------------------------
for i in range(n, len(axes)):
    fig.delaxes(axes[i])

# --------------------------------------------------
# Shared labels and title
# --------------------------------------------------
fig.text(0.5, 0.04, "Date", ha="center", fontsize=12)
fig.text(
    0.04, 0.5,
    "East–West Displacement [mm]",
    va="center", rotation="vertical", fontsize=12
)

fig.suptitle(
    "East–West Displacement Time Series\nPOIs",
    fontsize=16
)

plt.tight_layout(rect=[0.03, 0.06, 1, 0.93])

plt.savefig(
    "SBAS_Island2_EW_TimeSeries_BesidesWall_Subplots_Y21_25_B48_13Feb.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# --------------------------------------------------
# Load LOS displacement cubes
# --------------------------------------------------


# --------------------------------------------------
# Correct vertical solver
# --------------------------------------------------
def solve_vertical(los_a, los_d, le_a, lu_a, le_d, lu_d):

    los_d = los_d.interp(date=los_a.date)

    det = lu_a * le_d - le_a * lu_d
    if abs(det) < 1e-8:
        return None

    up = (le_d * los_a - le_a * los_d) / det
    return up.astype("float64")

# --------------------------------------------------
# Extract full-resolution vertical time series
# --------------------------------------------------
vertical_dict = {}
xy_dict = {}

for (_, ra), (_, rd) in zip(poi_geom_asc.iterrows(), poi_geom_desc.iterrows()):

    name = ra["point"]

    xa, ya = ra["x"], ra["y"]
    xd, yd = rd["x"], rd["y"]

    los_a = los_asc.sel(x=xa, y=ya, method="nearest").squeeze()
    los_d = los_desc.sel(x=xd, y=yd, method="nearest").squeeze()

    vertical_ts = solve_vertical(
        los_a, los_d,
        ra["look_E"], ra["look_U"],
        rd["look_E"], rd["look_U"]
    )

    if vertical_ts is not None:
        vertical_dict[name] = vertical_ts
        xy_dict[name] = (xa, ya)

if not vertical_dict:
    raise RuntimeError("No vertical data computed")

# --------------------------------------------------
# Zero reference to first valid epoch
# --------------------------------------------------
vertical_rel_dict = {}

for name, ts in vertical_dict.items():
    s = ts.to_pandas()
    first_valid = s.first_valid_index()

    if first_valid is not None:
        vertical_rel_dict[name] = s - s.loc[first_valid]

# --------------------------------------------------
# Use full time resolution
# --------------------------------------------------
dates = vertical_rel_dict[next(iter(vertical_rel_dict))].index

# Robust magnitude scaling
all_vals = np.concatenate([
    np.abs(ts.dropna().values)
    for ts in vertical_rel_dict.values()
])

ref_mag = np.percentile(all_vals, 90)

# --------------------------------------------------
# Animation setup
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 6), dpi=150)

VIS_SCALE = 1.0
MIN_LEN = 0.2
MAX_LEN = 1.5
QUIVER_SCALE = 10
FRAME_INTERVAL = 800   # faster since more frames

# --------------------------------------------------
# Animation update
# --------------------------------------------------
def update(frame):

    ax.clear()
    date = dates[frame]

    Xu, Yu, Uu, Vu = [], [], [], []
    Xs, Ys, Us, Vs = [], [], [], []

    for name, ts in vertical_rel_dict.items():

        if date not in ts.index:
            continue

        val = ts.loc[date]
        if not np.isfinite(val):
            continue

        x, y = xy_dict[name]
        mag = np.clip(abs(val) / ref_mag, MIN_LEN, MAX_LEN)

        if val >= 0:
            Xu.append(x); Yu.append(y)
            Uu.append(0.0); Vu.append(+mag * VIS_SCALE)
        else:
            Xs.append(x); Ys.append(y)
            Us.append(0.0); Vs.append(-mag * VIS_SCALE)

    ax.scatter(Xu + Xs, Yu + Ys, s=6, color="k", alpha=0.4, zorder=1)

    if Xu:
        ax.quiver(
            Xu, Yu, Uu, Vu,
            angles="xy",
            scale_units="width",
            scale=QUIVER_SCALE,
            width=0.005,
            pivot="middle",
            color="red",
            zorder=3
        )

    if Xs:
        ax.quiver(
            Xs, Ys, Us, Vs,
            angles="xy",
            scale_units="width",
            scale=QUIVER_SCALE,
            width=0.005,
            pivot="middle",
            color="blue",
            zorder=3
        )

    ax.set_aspect("auto")
    ax.set_title(f"Vertical Displacement  {date.strftime('%Y-%m-%d')}", fontsize=14)
    ax.set_xlabel("X coordinate")
    ax.set_ylabel("Y coordinate")
    ax.grid(True, linestyle="--", alpha=0.3)

# --------------------------------------------------
# Create animation
# --------------------------------------------------
anim = FuncAnimation(
    fig,
    update,
    frames=len(dates),
    interval=FRAME_INTERVAL,
    repeat=True
)

gif_path = "SBAS_Island5_Vertical_AllEpochs_Y21_25_B36.gif"

anim.save(
    gif_path,
    writer=PillowWriter(fps=1.5)
)

plt.close(fig)

print("Animated GIF saved to:", gif_path)


In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# --------------------------------------------------
# Load LOS displacement cubes
# --------------------------------------------------

# --------------------------------------------------
# Correct vertical + east solver
# --------------------------------------------------
def solve_vertical_east(los_a, los_d, le_a, lu_a, le_d, lu_d):

    los_d = los_d.interp(date=los_a.date)

    det = lu_a * le_d - le_a * lu_d
    if abs(det) < 1e-8:
        return None, None

    up = (le_d * los_a - le_a * los_d) / det
    east = (-lu_d * los_a + lu_a * los_d) / det

    return up.astype("float64"), east.astype("float64")

# --------------------------------------------------
# Extract full-resolution East time series
# --------------------------------------------------
east_dict = {}
xy_dict = {}

for (_, ra), (_, rd) in zip(poi_geom_asc.iterrows(), poi_geom_desc.iterrows()):

    name = ra["point"]

    xa, ya = ra["x"], ra["y"]
    xd, yd = rd["x"], rd["y"]

    los_a = los_asc.sel(x=xa, y=ya, method="nearest").squeeze()
    los_d = los_desc.sel(x=xd, y=yd, method="nearest").squeeze()

    up_ts, east_ts = solve_vertical_east(
        los_a, los_d,
        ra["look_E"], ra["look_U"],
        rd["look_E"], rd["look_U"]
    )

    if east_ts is not None:
        east_dict[name] = east_ts
        xy_dict[name] = (xa, ya)

if not east_dict:
    raise RuntimeError("No East-West data computed")

# --------------------------------------------------
# Zero reference to first valid epoch
# --------------------------------------------------
east_rel_dict = {}

for name, ts in east_dict.items():
    s = ts.to_pandas()
    first_valid = s.first_valid_index()

    if first_valid is not None:
        east_rel_dict[name] = s - s.loc[first_valid]

# --------------------------------------------------
# Use full time resolution
# --------------------------------------------------
dates = east_rel_dict[next(iter(east_rel_dict))].index

# Robust magnitude scaling
all_vals = np.concatenate([
    np.abs(ts.dropna().values)
    for ts in east_rel_dict.values()
])

ref_mag = np.percentile(all_vals, 90)

# --------------------------------------------------
# Animation setup
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 6), dpi=150)

VIS_SCALE = 1.0
MIN_LEN = 0.2
MAX_LEN = 1.5
QUIVER_SCALE = 10
FRAME_INTERVAL = 800

# --------------------------------------------------
# Animation update
# --------------------------------------------------
def update(frame):

    ax.clear()
    date = dates[frame]

    Xe, Ye, Ue, Ve = [], [], [], []
    Xw, Yw, Uw, Vw = [], [], [], []

    for name, ts in east_rel_dict.items():

        if date not in ts.index:
            continue

        val = ts.loc[date]
        if not np.isfinite(val):
            continue

        x, y = xy_dict[name]
        mag = np.clip(abs(val) / ref_mag, MIN_LEN, MAX_LEN)

        if val >= 0:
            Xe.append(x); Ye.append(y)
            Ue.append(+mag * VIS_SCALE); Ve.append(0.0)
        else:
            Xw.append(x); Yw.append(y)
            Uw.append(-mag * VIS_SCALE); Vw.append(0.0)

    ax.scatter(Xe + Xw, Ye + Yw, s=6, color="k", alpha=0.4, zorder=1)

    if Xe:
        ax.quiver(
            Xe, Ye, Ue, Ve,
            angles="xy",
            scale_units="width",
            scale=QUIVER_SCALE,
            width=0.005,
            pivot="middle",
            color="red",
            zorder=3
        )

    if Xw:
        ax.quiver(
            Xw, Yw, Uw, Vw,
            angles="xy",
            scale_units="width",
            scale=QUIVER_SCALE,
            width=0.005,
            pivot="middle",
            color="blue",
            zorder=3
        )

    ax.set_aspect("auto")
    ax.set_title(f"East-West Displacement  {date.strftime('%Y-%m-%d')}", fontsize=14)
    ax.set_xlabel("X coordinate")
    ax.set_ylabel("Y coordinate")
    ax.grid(True, linestyle="--", alpha=0.3)

# --------------------------------------------------
# Create animation
# --------------------------------------------------
anim = FuncAnimation(
    fig,
    update,
    frames=len(dates),
    interval=FRAME_INTERVAL,
    repeat=True
)

gif_path = "SBAS_Island5_EastWest_AllEpochs_Y21_25_B36.gif"

anim.save(
    gif_path,
    writer=PillowWriter(fps=1.5)
)

plt.close(fig)

print("Animated GIF saved to:", gif_path)


In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# --------------------------------------------------
# Load LOS displacement cubes
# --------------------------------------------------
# --------------------------------------------------
# Correct Vertical solver (consistent with your EW solver)
# --------------------------------------------------
def solve_vertical(los_a, los_d, le_a, lu_a, le_d, lu_d):

    los_d = los_d.interp(date=los_a.date)

    det = lu_a * le_d - le_a * lu_d
    if abs(det) < 1e-8:
        return None

    up = (le_d * los_a - le_a * los_d) / det
    return up.astype("float64")

# --------------------------------------------------
# Extract Vertical time series
# --------------------------------------------------
vertical_dict = {}
xy_dict = {}

for (_, ra), (_, rd) in zip(poi_geom_asc.iterrows(), poi_geom_desc.iterrows()):

    name = ra["point"]

    xa, ya = ra["x"], ra["y"]
    xd, yd = rd["x"], rd["y"]

    los_a = los_asc.sel(x=xa, y=ya, method="nearest").squeeze()
    los_d = los_desc.sel(x=xd, y=yd, method="nearest").squeeze()

    vertical_ts = solve_vertical(
        los_a, los_d,
        ra["look_E"], ra["look_U"],
        rd["look_E"], rd["look_U"]
    )

    if vertical_ts is not None:
        vertical_dict[name] = vertical_ts
        xy_dict[name] = (xa, ya)

# --------------------------------------------------
# Zero reference to first valid epoch
# --------------------------------------------------
vertical_rel_dict = {}

for name, ts in vertical_dict.items():
    s = ts.to_pandas()
    first_valid = s.first_valid_index()

    if first_valid is not None:
        vertical_rel_dict[name] = s - s.loc[first_valid]

# --------------------------------------------------
# Monthly cumulative displacement
# --------------------------------------------------
vertical_monthly_dict = {}

for name, s in vertical_rel_dict.items():
    m = s.resample("M").sum()

    if not m.isna().all():
        vertical_monthly_dict[name] = m

if not vertical_monthly_dict:
    raise RuntimeError("No monthly vertical data available")

# --------------------------------------------------
# Robust magnitude scaling reference
# --------------------------------------------------
all_vals = np.concatenate([
    np.abs(ts.dropna().values)
    for ts in vertical_monthly_dict.values()
])

ref_mag = np.percentile(all_vals, 90)

dates = vertical_monthly_dict[next(iter(vertical_monthly_dict))].index

# --------------------------------------------------
# Animation setup
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 6), dpi=150)

VIS_SCALE = 1.0
MIN_LEN = 0.2
MAX_LEN = 1.5
QUIVER_SCALE = 10
FRAME_INTERVAL = 1200

# --------------------------------------------------
# Animation update
# --------------------------------------------------
def update(frame):

    ax.clear()
    date = dates[frame]

    Xu, Yu, Uu, Vu = [], [], [], []
    Xs, Ys, Us, Vs = [], [], [], []

    for name, ts in vertical_monthly_dict.items():

        if date not in ts.index:
            continue

        val = ts.loc[date]

        if not np.isfinite(val):
            continue

        x, y = xy_dict[name]
        mag = np.clip(abs(val) / ref_mag, MIN_LEN, MAX_LEN)

        if val >= 0:
            Xu.append(x); Yu.append(y)
            Uu.append(0.0); Vu.append(+mag * VIS_SCALE)
        else:
            Xs.append(x); Ys.append(y)
            Us.append(0.0); Vs.append(-mag * VIS_SCALE)

    ax.scatter(Xu + Xs, Yu + Ys, s=6, color="k", alpha=0.4, zorder=1)

    if Xu:
        ax.quiver(
            Xu, Yu, Uu, Vu,
            angles="xy",
            scale_units="width",
            scale=QUIVER_SCALE,
            width=0.005,
            pivot="middle",
            color="red",
            zorder=3
        )

    if Xs:
        ax.quiver(
            Xs, Ys, Us, Vs,
            angles="xy",
            scale_units="width",
            scale=QUIVER_SCALE,
            width=0.005,
            pivot="middle",
            color="blue",
            zorder=3
        )

    ax.set_aspect("auto")
    ax.set_title(f"Vertical Displacement  {date.strftime('%Y-%m')}", fontsize=14)
    ax.set_xlabel("X coordinate")
    ax.set_ylabel("Y coordinate")
    ax.grid(True, linestyle="--", alpha=0.3)

# --------------------------------------------------
# Create animation
# --------------------------------------------------
anim = FuncAnimation(
    fig,
    update,
    frames=len(dates),
    interval=FRAME_INTERVAL,
    repeat=True
)

gif_path = "SBAS_Island2_Vertical_Subsidence_Uplift_B48_Y21_25_Corrected.gif"

anim.save(
    gif_path,
    writer=PillowWriter(fps=0.8)
)

plt.close(fig)

print("Animated GIF saved to:", gif_path)


In [ ]:
import math
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# --------------------------------------------------
# Load LOS displacement cubes
# --------------------------------------------------
#los_asc  = xr.open_dataarray("/scratch/pm4167/ADNOC/disp_ps_ADIsland2_2021_24_Asc_B36_New3.nc")
#los_desc = xr.open_dataarray("/scratch/pm4167/ADNOC/disp_ps_ADIsland2_2021_24_Desc_B36_New4.nc")

# --------------------------------------------------
# Load POI geometry with look vectors
# --------------------------------------------------
#poi_geom_asc  = pd.read_csv("/scratch/pm4167/ADNOC/poi_geometryALLPoints_angles_AD_island2_2021_24_ASC_B48_New4_1.csv")
#poi_geom_desc = pd.read_csv("/scratch/pm4167/ADNOC/poi_geometryALLPoints_angles_AD_island2_2021_24_DESC_B36_New4.csv")

# --------------------------------------------------
# Vertical solver (Vertical + East system)
# --------------------------------------------------
def solve_vertical(los_a, los_d, le_a, lu_a, le_d, lu_d):
    los_d = los_d.interp(date=los_a.date)

    det = le_a * lu_d - lu_a * le_d
    if abs(det) < 1e-8:
        return None

    vertical = (le_d * los_a - le_a * los_d) / det
    return vertical.astype("float64")

# --------------------------------------------------
# Extract Vertical time series
# --------------------------------------------------
vertical_dict = {}
xy_dict = {}

for (_, ra), (_, rd) in zip(poi_geom_asc.iterrows(), poi_geom_desc.iterrows()):
    name = ra["point"]

    xa, ya = ra["x"], ra["y"]
    xd, yd = rd["x"], rd["y"]

    los_a = los_asc.sel(x=xa, y=ya, method="nearest").squeeze()
    los_d = los_desc.sel(x=xd, y=yd, method="nearest").squeeze()

    vertical_ts = solve_vertical(
        los_a, los_d,
        ra["look_E"], ra["look_U"],
        rd["look_E"], rd["look_U"]
    )

    if vertical_ts is not None:
        vertical_dict[name] = vertical_ts
        xy_dict[name] = (xa, ya)

# --------------------------------------------------
# Zero reference to first valid epoch
# --------------------------------------------------
vertical_rel_dict = {}

for name, ts in vertical_dict.items():
    s = ts.to_pandas()
    first_valid = s.first_valid_index()
    if first_valid is not None:
        vertical_rel_dict[name] = s - s.loc[first_valid]

# --------------------------------------------------
# Monthly cumulative displacement
# --------------------------------------------------
vertical_monthly_dict = {}

for name, s in vertical_rel_dict.items():
    m = s.resample("M").sum()
    if not m.isna().all():
        vertical_monthly_dict[name] = m

if not vertical_monthly_dict:
    raise RuntimeError("No monthly vertical data available")

# --------------------------------------------------
# Robust magnitude reference for scaling
# --------------------------------------------------
all_vals = np.concatenate([
    np.abs(ts.dropna().values)
    for ts in vertical_monthly_dict.values()
])

ref_mag = np.percentile(all_vals, 90)

dates = vertical_monthly_dict[next(iter(vertical_monthly_dict))].index

fig, ax = plt.subplots(figsize=(6, 6), dpi=150)

VIS_SCALE = 1.0
MIN_LEN = 0.2
MAX_LEN = 1.5
QUIVER_SCALE = 10
FRAME_INTERVAL = 1200

# --------------------------------------------------
# Animation update
# --------------------------------------------------
def update(frame):
    ax.clear()
    date = dates[frame]

    Xu, Yu, Uu, Vu = [], [], [], []
    Xs, Ys, Us, Vs = [], [], [], []

    for name, ts in vertical_monthly_dict.items():
        if date not in ts.index:
            continue

        val = ts.loc[date]
        if not np.isfinite(val):
            continue

        x, y = xy_dict[name]
        mag = np.clip(abs(val) / ref_mag, MIN_LEN, MAX_LEN)

        if val >= 0:
            Xu.append(x); Yu.append(y)
            Uu.append(0.0); Vu.append(+mag * VIS_SCALE)
        else:
            Xs.append(x); Ys.append(y)
            Us.append(0.0); Vs.append(-mag * VIS_SCALE)

    ax.scatter(Xu + Xs, Yu + Ys, s=6, color="k", alpha=0.4, zorder=1)

    if Xu:
        ax.quiver(
            Xu, Yu, Uu, Vu,
            angles="xy",
            scale_units="width",
            scale=QUIVER_SCALE,
            width=0.005,
            pivot="middle",
            color="red",
            zorder=3
        )

    if Xs:
        ax.quiver(
            Xs, Ys, Us, Vs,
            angles="xy",
            scale_units="width",
            scale=QUIVER_SCALE,
            width=0.005,
            pivot="middle",
            color="blue",
            zorder=3
        )

    ax.set_aspect("auto")
    ax.set_title(f"Vertical Displacement  {date.strftime('%Y-%m')}", fontsize=14)
    ax.set_xlabel("X coordinate")
    ax.set_ylabel("Y coordinate")
    ax.grid(True, linestyle="--", alpha=0.3)

# --------------------------------------------------
# Create animation
# --------------------------------------------------
anim = FuncAnimation(
    fig,
    update,
    frames=len(dates),
    interval=FRAME_INTERVAL,
    repeat=True
)

gif_path = "SBAS_Island2_Vertical_Subsidence_Uplift_B48_Y21_25.gif"

anim.save(
    gif_path,
    writer=PillowWriter(fps=0.8)
)

plt.close(fig)

print(f"Animated GIF saved to: {gif_path}")
